# Timestral 101: Lesson 6 lab, solutions notebook

The complete code for the Lesson 6 lab (Kestrel Cold Chain), ready to run from top to bottom.
Run it locally (Python 3.10 or later) or in Google Colab. The first cells install Timestral and download the site file if it isn't next to the notebook yet.
Your numbers should match the ones in the course.

In [ ]:
%pip install -q git+https://github.com/mistralai/timestral.git pandas

In [ ]:
import os, urllib.request
if not os.path.exists("lab-kestrel-site.csv"):
    urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1n257-mHExrszrZAov2cGclGUaaC52td3", "lab-kestrel-site.csv")
print(os.path.getsize("lab-kestrel-site.csv"), "bytes")

## Guided code walkthrough

> **Banner(code-bracket):** The complete code for this lab is also in a ready-to-run solutions notebook: download it (Google Drive, then **File → Download**), or open it in Google Colab to run it in your browser with nothing to install. Work through the steps yourself first, and use the notebook to check your results.

Download lab-kestrel-site.csv (Google Drive, then **File → Download**) and save it next to your notebook or script. The data is synthetic. It stands in for five refrigeration meters at one Kestrel site, exported the way Kestrel's export works, including the dropouts written as 0. Put each step in a notebook cell, or keep adding to one script.

**Step 1: Imports and a helper.** This version of the lab helper keeps the array's shape, because you'll now forecast several meters at once.

In [ ]:
import time
import numpy as np
import pandas as pd
from timestral import TimestralForecaster


def to_np(x):
    """Turn a NumPy array or torch tensor into a NumPy array, keeping its shape."""
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

**Step 2: Load the site file and reshape it for Timestral.** The file is in long format: one row per meter per 15-minute step, with the columns `timestamp`, `meter_id`, `load_kw` and `defrost_scheduled`. Pivot it so each meter becomes one series, shaped `(series, variates, time)`.

In [ ]:
STEPS_PER_DAY = 96  # 15-minute readings
HORIZON = 96  # tomorrow

df = pd.read_csv("lab-kestrel-site.csv", parse_dates=["timestamp"])

load_wide = df.pivot(
    index="meter_id", columns="timestamp", values="load_kw"
).sort_index()
defrost_wide = df.pivot(
    index="meter_id", columns="timestamp", values="defrost_scheduled"
).sort_index()

meters = list(load_wide.index)
S, N = load_wide.shape  # meters, time steps

# (series, variates, time)
raw = load_wide.to_numpy(dtype=np.float32).reshape(S, 1, N).copy()
# known a day ahead
defrost = defrost_wide.to_numpy(dtype=np.float32).reshape(S, 1, N).copy()
print(meters, raw.shape)

Each meter here is its own series with one variate. When a single series has several variates, Timestral forecasts them jointly.

**Step 3: Fix the input you found in Part 1.** Zeros are real readings to the model, so turn the dropouts into `NaN`. Then count them per meter.

In [ ]:
clean = np.where(raw == 0.0, np.nan, raw).astype(np.float32)

dropouts = np.isnan(clean).sum(axis=(1, 2))
for m, n in zip(meters, dropouts):
    print(f"{m}: {n} dropouts marked missing")

Write down each meter's count. If one meter has far more gaps than the others, mention it in your note: it's a data-quality point worth raising with the customer.

**Step 4: Score the customer's current method.** The planner copies the same day from last week, so that copy is their plan. The scoring helpers work on all five meters at once, and they also keep each meter's under-buy rate, so one bad meter can't hide behind the average.

In [ ]:
cutoffs = [d * STEPS_PER_DAY for d in range(49, 56)]  # the last 7 days


def score_day(point, plan, actual, p10=None, p90=None):
    """Score one cut-off across all meters. Each array is (meters, 96)."""
    ok = ~np.isnan(actual) & ~np.isnan(plan)
    under = np.where(ok, actual > plan, np.nan)
    return {
        "mae": np.mean(np.abs(point[ok] - actual[ok])),
        "coverage": (
            np.nan
            if p10 is None
            else np.mean((actual[ok] >= p10[ok]) & (actual[ok] <= p90[ok]))
        ),
        "under": np.nanmean(under),
        "headroom": np.mean(plan[ok] - actual[ok]),
        "under_by_meter": np.nanmean(under, axis=1),
    }


def summarise(rows, ms=np.nan):
    out = {
        k: np.mean([r[k] for r in rows])
        for k in ["mae", "coverage", "under", "headroom"]
    }
    out["worst"] = np.max(np.mean([r["under_by_meter"] for r in rows], axis=0))
    out["ms"] = ms
    return out


def score_baseline(cutoffs):
    rows = []
    for c in cutoffs:
        week_ago = c - 7 * STEPS_PER_DAY
        plan = clean[:, 0, week_ago : week_ago + HORIZON]  # a single number has no band
        actual = clean[:, 0, c : c + HORIZON]
        rows.append(score_day(plan, plan, actual))
    return summarise(rows)

**Step 5: Backtest a Timestral size on the same cut-offs.** Each forecast uses only data from before its cut-off. The defrost schedule runs through the forecast day. One call forecasts all five meters: the context is `(5, 1, c)`, the known covariates are `(5, 1, c + 96)`, and the quantiles come back as `(5, 1, 96, 21)`.

In [ ]:
def score_timestral(model, cutoffs):
    def run(c):
        return model.forecast(
            clean[:, :, :c],
            prediction_length=HORIZON,
            known_covariates=defrost[:, :, : c + HORIZON],
        )

    run(cutoffs[0])  # warm-up call, so one-off setup doesn't count in the timings
    rows, times = [], []
    for c in cutoffs:
        start = time.perf_counter()
        fc = run(c)  # all five meters in one batched call
        times.append((time.perf_counter() - start) * 1000)
        p10, p50, p90 = (
            to_np(q).reshape(S, HORIZON)
            for q in (fc.quantile(0.1), fc.median, fc.quantile(0.9))
        )
        actual = clean[:, 0, c : c + HORIZON]
        rows.append(score_day(p50, p90, actual, p10, p90))  # plan = P90, as agreed
    return summarise(rows, ms=np.mean(times))

## Run it and see

**Step 6: Run all three contenders and print one table.** If you didn't download the 209M in Lesson 2, it's an 837 MB download.

In [ ]:
results = {"Current method": score_baseline(cutoffs)}
for name in ["mistralai/Timestral-209M", "mistralai/Timestral-28M"]:
    model = TimestralForecaster.from_pretrained(name)
    results[name.split("-")[-1]] = score_timestral(model, cutoffs)


def fmt(v, f):
    return "-" if np.isnan(v) else format(v, f)


print(
    f"{'':16}{'median err':>11}{'P10-P90':>9}{'under-buy':>10}"
    f"{'worst meter':>12}{'headroom':>10}{'ms':>7}"
)
for name, r in results.items():
    print(
        f"{name:16}{fmt(r['mae'], '.1f'):>11}{fmt(r['coverage'], '.0%'):>9}"
        f"{fmt(r['under'], '.0%'):>10}{fmt(r['worst'], '.0%'):>12}"
        f"{fmt(r['headroom'], '.1f'):>10}{fmt(r['ms'], '.0f'):>7}"
    )

**How to read your table**

- **Median error** is the average distance, in kW, between the median (or the planner's copy) and what happened, across all five meters. Lower is better.
- **P10–P90** is how often actuals landed inside the 80% band. Over seven days, somewhere near 80% is healthy.
- **Under-buy** is how often actual use beat the plan, across all five meters. This is the penalty the customer cares about. Compare it with the 10% target.
- **Worst meter** is the under-buy rate of the meter that under-bought most often. The target applies to every pilot meter, so check this column too.
- **Headroom** is how far, on average, the plan sat above actual use. That's roughly what the safety margin costs them.
- **ms** is your machine's time for one batched call covering all five meters. It's only a rough guide to their nightly run of about 560 meters. Their CPU servers will give different numbers, so time it there during the pilot.

[Background: gray]